# Loan Default Risk — Exploratory Data Analysis

**Project 4 | Gwachat Kozah**

This notebook explores the three SuperLender datasets before any modelling takes place. The goal is to understand the data structure, the target variable distribution, missing values, and the behavioural patterns in prior loan history that are likely to drive default risk.

The three tables we work with are:
- `trainperf.csv` — the loan to be predicted, including the target `good_bad_flag`
- `traindemographics.csv` — customer demographic information
- `trainprevloans.csv` — all prior loans taken by each customer

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Allow imports from src/
sys.path.append(os.path.abspath(".."))

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")

%matplotlib inline

## 1. Load Raw Data

In [ ]:
from src.loader import load_raw_tables

perf, demo, prev = load_raw_tables(split="train")

print(f"trainperf:      {perf.shape}")
print(f"traindemographics: {demo.shape}")
print(f"trainprevloans: {prev.shape}")

## 2. Performance Data — Target Variable

In [ ]:
perf.head()

In [ ]:
perf.info()

In [ ]:
# Missing values
perf.isnull().sum()

In [ ]:
# Target variable distribution
target_counts = perf["good_bad_flag"].value_counts()
target_pct = perf["good_bad_flag"].value_counts(normalize=True) * 100

print(target_counts)
print(target_pct.round(1))

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], edgecolor="white")
ax.set_title("Target Variable Distribution")
ax.set_xlabel("Loan Outcome")
ax.set_ylabel("Count")
ax.set_xticklabels(["Good", "Bad"], rotation=0)
for i, v in enumerate(target_counts):
    ax.text(i, v + 20, f"{target_pct.iloc[i]:.1f}%", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("../outputs/figures/target_distribution.png", dpi=150)
plt.show()

In [ ]:
# Loan amount and term distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

perf["loanamount"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Loan Amount Distribution")
axes[0].set_xlabel("Loan Amount")

perf["totaldue"].hist(bins=40, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Total Due Distribution")
axes[1].set_xlabel("Total Due")

perf["termdays"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color="steelblue", edgecolor="white")
axes[2].set_title("Loan Term (Days)")
axes[2].set_xlabel("Term Days")

plt.tight_layout()
plt.savefig("../outputs/figures/loan_distributions.png", dpi=150)
plt.show()

In [ ]:
# Loan amount by target class
fig, ax = plt.subplots(figsize=(8, 4))
perf.boxplot(column="loanamount", by="good_bad_flag", ax=ax)
ax.set_title("Loan Amount by Default Status")
ax.set_xlabel("Loan Outcome")
ax.set_ylabel("Loan Amount")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 3. Demographic Data

In [ ]:
demo.head()

In [ ]:
demo.info()

In [ ]:
# Missing values as percentages
missing = (demo.isnull().sum() / len(demo) * 100).round(1)
print(missing[missing > 0])

In [ ]:
# Categorical feature distributions
cat_cols = ["bank_account_type", "employment_status_clients", "level_of_education_clients"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    counts = demo[col].value_counts(dropna=False)
    counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("../outputs/figures/demographic_distributions.png", dpi=150)
plt.show()

In [ ]:
# Age distribution — merge perf dates to compute age at application
demo_perf = perf[["customerid", "approveddate"]].merge(demo[["customerid", "birthdate"]], on="customerid")
demo_perf["approveddate"] = pd.to_datetime(demo_perf["approveddate"])
demo_perf["birthdate"] = pd.to_datetime(demo_perf["birthdate"])
demo_perf["age"] = ((demo_perf["approveddate"] - demo_perf["birthdate"]).dt.days / 365.25).round(0)

fig, ax = plt.subplots(figsize=(8, 4))
demo_perf["age"].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Customer Age at Loan Approval")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../outputs/figures/age_distribution.png", dpi=150)
plt.show()

print(demo_perf["age"].describe())

## 4. Previous Loans Data — Behavioural Patterns

In [ ]:
prev.head()

In [ ]:
prev.info()

In [ ]:
# How many prior loans does each customer have?
loan_counts = prev.groupby("customerid")["systemloanid"].count()

fig, ax = plt.subplots(figsize=(8, 4))
loan_counts.hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Number of Prior Loans per Customer")
ax.set_xlabel("Prior Loan Count")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.savefig("../outputs/figures/prior_loan_counts.png", dpi=150)
plt.show()

print(loan_counts.describe())

In [ ]:
# Compute days late on first payment for each previous loan
prev_dates = prev.copy()
prev_dates["firstduedate"] = pd.to_datetime(prev_dates["firstduedate"])
prev_dates["firstrepaiddate"] = pd.to_datetime(prev_dates["firstrepaiddate"])
prev_dates["days_late"] = (prev_dates["firstrepaiddate"] - prev_dates["firstduedate"]).dt.days

fig, ax = plt.subplots(figsize=(8, 4))
prev_dates["days_late"].clip(-30, 60).hist(bins=50, ax=ax, color="steelblue", edgecolor="white")
ax.axvline(0, color="tomato", linestyle="--", label="On time")
ax.set_title("Days Late on First Payment (Prior Loans)")
ax.set_xlabel("Days Late (negative = paid early)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/days_late_distribution.png", dpi=150)
plt.show()

late_pct = (prev_dates["days_late"] > 0).mean() * 100
print(f"Percentage of prior loans paid late: {late_pct:.1f}%")

## 5. New vs Repeat Customers

In [ ]:
# Check how many customers in trainperf have prior loan history
repeat_customers = set(prev["customerid"].unique())
all_customers = set(perf["customerid"].unique())
new_customers = all_customers - repeat_customers

print(f"Total customers in trainperf:       {len(all_customers)}")
print(f"Customers with prior loan history:  {len(repeat_customers & all_customers)}")
print(f"New customers (no prior history):   {len(new_customers)}")
print(f"\nThis dataset is {(1 - len(new_customers)/len(all_customers))*100:.1f}% repeat customers.")

## 6. Behavioural Features vs Target

In [ ]:
# Engineer prevloans features and merge with target to explore relationships
from src.features import engineer_prevloans_features

prev_features = engineer_prevloans_features(prev)
perf_with_target = perf[["customerid", "good_bad_flag"]].copy()
merged = perf_with_target.merge(prev_features, on="customerid", how="left")

merged.head()

In [ ]:
# Late payment rate by target class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

merged.boxplot(column="prev_late_payment_rate", by="good_bad_flag", ax=axes[0])
axes[0].set_title("Prior Late Payment Rate by Outcome")
axes[0].set_xlabel("Loan Outcome")
axes[0].set_ylabel("Late Payment Rate")
plt.sca(axes[0])
plt.suptitle("")

merged.boxplot(column="prev_avg_days_late", by="good_bad_flag", ax=axes[1])
axes[1].set_title("Average Days Late (Prior Loans) by Outcome")
axes[1].set_xlabel("Loan Outcome")
axes[1].set_ylabel("Avg Days Late")
plt.suptitle("")

plt.tight_layout()
plt.savefig("../outputs/figures/behavioural_vs_target.png", dpi=150)
plt.show()

In [ ]:
# Correlation heatmap of behavioural features
behavioural_cols = [
    "prev_loan_count", "prev_avg_loanamount", "prev_avg_days_late",
    "prev_late_payment_rate", "prev_total_late_payments",
    "prev_avg_interest_ratio", "prev_avg_loan_duration"
]

fig, ax = plt.subplots(figsize=(10, 8))
corr = merged[behavioural_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation — Behavioural Features")
plt.tight_layout()
plt.savefig("../outputs/figures/behavioural_correlation.png", dpi=150)
plt.show()

## 7. Summary of Key EDA Findings

*Fill in after running the notebook — key observations about class imbalance, missing values, behavioural patterns, and any surprises in the data.*